# Qualitative examples for the paper

Goal: surface a small number of concrete query-level examples to illustrate each of the three
main findings (rerankers, gene-aware query expansion, full-text retrieval).

Design principle: each example is an **existence proof of a mechanism**, not evidence for the
population. Population evidence lives in Table 1 / Fig 4 / Fig 5 / Table S1 / Fig S2 / Fig S4.
We pick examples that make the mechanism legible in a small table cell; selection is
transparent (filter rules are in the cells below).

**Outputs:** the final cells produce short markdown snippets that can be pasted into
`output/paper_figures/paper.tex`. Body-chunk text lookup for Ex.3 is provided as a helper.

In [44]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable

import polars as pl

REPO = Path("/Users/yun/develop/dictycite")
OUT = REPO / "output"

## 1. Paths

All three findings can be reproduced from the public goldset (`7a`) and the
query-expansion benchmark (`7d`). For each finding, we point to the workflow whose
`rerank/post_rerank_fusion/runs/*.tsv` gives the final ranking under that system.

In [45]:
GOLDSET_7A = OUT / "dicty_gold_build" / "7a_dicty_gold_llm_public.jsonl"
QE_BENCH_7D = OUT / "dicty_gold_build" / "7d_dicty_gold_query_expansion_benchmark.jsonl"
ABSTRACTS_PARQUET = OUT / "dicty_gold_build" / "3_articles_cleaned_abstract.parquet"
CHUNKS_JSONL = OUT / "pdf_extraction" / "v2" / "chunks.jsonl"

# Ex.1 — reranker comparison (abstract corpus, full 1,656-query goldset)
RUN_BM25_7A = (
    OUT
    / "workflow_frida_7a_public_goldset_rerank_gemma"
    / "retrieval/bm25/runs/BM25__7a_dicty_gold_llm_public__top5000.tsv"
)
RUN_MSMARCO_7A = (
    OUT
    / "workflow_frida_7a_public_goldset_rerank_ms_marco_minilm"
    / "rerank/post_rerank_fusion/runs"
    / "best_rrf_7a_dicty_gold_llm_public_top5000_rrf_poolR50_poolH50_k60.tsv"
)
RUN_BGEM3_7A = (
    OUT
    / "workflow_frida_7a_public_goldset_both_routes"
    / "rerank/post_rerank_fusion/runs"
    / "best_rrf_7a_dicty_gold_llm_public_top5000_rrf_poolR50_poolH50_k60.tsv"
)
RUN_MEDCPT_7A = (
    OUT
    / "workflow_vega_7a_public_goldset_rerank_medcpt"
    / "rerank/post_rerank_fusion/runs"
    / "best_rrf_7a_dicty_gold_llm_public_top5000_rrf_poolR50_poolH50_k60.tsv"
)
RUN_GEMMA_7A = (
    OUT
    / "workflow_frida_7a_public_goldset_rerank_gemma"
    / "rerank/post_rerank_fusion/runs"
    / "best_rrf_7a_dicty_gold_llm_public_top5000_rrf_poolR50_poolH50_k60.tsv"
)

# Ex.2 — QE rerank-query sweep (fixed deepest-expansion candidate pool, vary reranker query)
# Source dataset is 7d (563-query expansion benchmark). BGE-m3 sweep (matches Table S1 row).
RUN_QE_BGEM3_BODY = (
    OUT
    / "workflow_baseline_full_sweep/workflow_fixed_long_rerank_sweep_7d"
    / "fixed_long_rerank_sweep/rerank_body/runs"
    / "best_rrf_7d_dicty_gold_query_expansion_benchmark_top5000.tsv"
)
RUN_QE_BGEM3_SYN = (
    OUT
    / "workflow_baseline_full_sweep/workflow_fixed_long_rerank_sweep_7d"
    / "fixed_long_rerank_sweep/rerank_synonyms/runs"
    / "best_rrf_7d_dicty_gold_query_expansion_benchmark_top5000.tsv"
)
RUN_QE_BGEM3_LONG = (
    OUT
    / "workflow_baseline_full_sweep/workflow_fixed_long_rerank_sweep_7d"
    / "fixed_long_rerank_sweep/rerank_long/runs"
    / "best_rrf_7d_dicty_gold_query_expansion_benchmark_top5000.tsv"
)

# Ex.3 — abstract-only vs chunked full-text corpus (BGE-m3, 7a goldset)
RUN_ABS_BGEM3_7A = RUN_BGEM3_7A  # same as Ex.1 BGE-m3
RUN_CHUNK_BGEM3_7A = (
    OUT
    / "workflow_frida_7a_public_goldset_chunked_v2"
    / "rerank/post_rerank_fusion/runs"
    / "best_rrf_7a_dicty_gold_llm_public_top5000_rrf_poolR300_poolH300_k60.tsv"
)

# Sanity-check every path before we go further.
for p in [
    GOLDSET_7A, QE_BENCH_7D, ABSTRACTS_PARQUET, CHUNKS_JSONL,
    RUN_BM25_7A, RUN_MSMARCO_7A, RUN_BGEM3_7A, RUN_MEDCPT_7A, RUN_GEMMA_7A,
    RUN_QE_BGEM3_BODY, RUN_QE_BGEM3_SYN, RUN_QE_BGEM3_LONG,
    RUN_ABS_BGEM3_7A, RUN_CHUNK_BGEM3_7A,
]:
    assert p.exists(), f"missing: {p}"
print("all paths ok")

all paths ok


## 2. Loaders

In [46]:
def load_goldset(path: Path) -> pl.DataFrame:
    """One row per query: qid, query_text, list of gold pmids, evidence_level (per doc), abstracts."""
    rows = []
    with open(path) as fh:
        for line in fh:
            r = json.loads(line)
            rows.append({
                "qid": int(r["query_id"]),
                "query_text": r["query_text"],
                "gold_pmids": [str(d["pmid"]) for d in r["docs"]],
                "gold_titles": [d["title"] for d in r["docs"]],
                "gold_abstracts": [d["abstract_clean"] for d in r["docs"]],
                "evidence_levels": [d["evidence_level"] for d in r["docs"]],
                "genes": r.get("genes", []),
            })
    return pl.DataFrame(rows)


def load_qe_benchmark(path: Path) -> pl.DataFrame:
    """One row per QE-eligible query: qid, original, +syn, +syn&prod, genes, gold pmids/titles."""
    rows = []
    with open(path) as fh:
        for line in fh:
            r = json.loads(line)
            rows.append({
                "qid": int(r["query_id"]),
                "query_text_body": r["query_text"],
                "query_text_synonyms": r.get("query_text_expansion_synonyms"),
                "query_text_long": r.get("query_text_synonym_products"),
                "gold_pmids": [str(d["pmid"]) for d in r["docs"]],
                "gold_titles": [d["title"] for d in r["docs"]],
                "gold_abstracts": [d["abstract_clean"] for d in r["docs"]],
                "detected_genes": r.get("query_gene_expansion", {}).get("detected_genes", []),
            })
    return pl.DataFrame(rows)


def load_run(path: Path) -> pl.DataFrame:
    """Run TSV: qid, docno, rank (and optionally score). Returns standardized cols,
    normalized to 1-indexed ranks (BM25 files are 0-indexed; rerank files are 1-indexed)."""
    df = pl.read_csv(path, separator="\t", schema_overrides={"qid": pl.Int64, "docno": pl.Utf8})
    df = df.select(["qid", "docno", "rank"])
    if df["rank"].min() == 0:
        df = df.with_columns((pl.col("rank") + 1).alias("rank"))
    return df


def rank_of_gold(run: pl.DataFrame, gold_pmids: list[str]) -> int | None:
    """Best (lowest) rank among gold pmids for this run-restricted-to-one-qid. None if missing."""
    matched = run.filter(pl.col("docno").is_in(gold_pmids))
    if matched.is_empty():
        return None
    return int(matched["rank"].min())

In [47]:
gold = load_goldset(GOLDSET_7A)
qe_bench = load_qe_benchmark(QE_BENCH_7D)
print(f"goldset: {len(gold)} queries; QE benchmark: {len(qe_bench)} queries")
gold.head(2)

goldset: 1656 queries; QE benchmark: 563 queries


qid,query_text,gold_pmids,gold_titles,gold_abstracts,evidence_levels,genes
i64,str,list[str],list[str],list[str],list[str],list[struct[4]]
1,"""A basic region in the tail is …","[""24747353""]","[""The association of myosin IB with actin waves in dictyostelium requires both the plasma membrane-binding site and actin-binding region in the myosin tail.""]","[""F-actin structures and their distribution are important determinants of the dynamic shapes and functions of eukaryotic cells. Actin waves are F-actin formations that move along the ventral cell membrane driven by actin polymerization. Dictyostelium myosin IB is associated with actin waves but its role in the wave is unknown. Myosin IB is a monomeric, non-filamentous myosin with a globular head that binds to F-actin and has motor activity, and a non-helical tail comprising a basic region, a glycine-proline-glutamine-rich region and an SH3-domain. The basic region binds to acidic phospholipids in the plasma membrane through a short basic-hydrophobic site and the Gly-Pro-Gln region binds F-actin. In the current work we found that both the basic-hydrophobic site in the basic region and the Gly-Pro-Gln region of the tail are required for the association of myosin IB with actin waves. This is the first evidence that the Gly-Pro-Gln region is required for localization of myosin IB to a specific actin structure in situ. The head is not required for myosin IB association with actin waves but binding of the head to F-actin strengthens the association of myosin IB with waves and stabilizes waves. Neither the SH3-domain nor motor activity is required for association of myosin IB with actin waves. We conclude that myosin IB contributes to anchoring actin waves to the plasma membranes by binding of the basic-hydrophobic site to acidic phospholipids in the plasma membrane and binding of the Gly-Pro-Gln region to F-actin in the wave.""]","[""abstract_supports_detail""]","[{""DDB_G0289117"",""myoB"",""DMIB, abmB, myoIB, myosin-1B, myosin IB"",""myosin IB heavy chain""}]"
10,"""A double mutant of gapA / rgaA…","[""20375144"", ""11447112""]","[""Involvement of the cytoskeleton in controlling leading-edge function during chemotaxis."", ""Recruitment of cortexillin into the cleavage furrow is controlled by Rac1 and IQGAP-related proteins.""]","[""In response to directional stimulation by a chemoattractant, cells rapidly activate a series of signaling pathways at the site closest to the chemoattractant source that leads to F-actin polymerization, pseudopod formation, and directional movement up the gradient. Ras proteins are major regulators of chemotaxis in Dictyostelium; they are activated at the leading edge, are required for chemoattractant-mediated activation of PI3K and TORC2, and are one of the most rapid responders, with activity peaking at approximately 3 s after stimulation. We demonstrate that in myosin II (MyoII) null cells, Ras activation is highly extended and is not restricted to the site closest to the chemoattractant source. This causes elevated, extended, and spatially misregulated activation of PI3K and TORC2 and their effectors Akt/PKB and PKBR1, as well as elevated F-actin polymerization. We further demonstrate that disruption of specific IQGAP/cortexillin complexes, which also regulate cortical mechanics, causes extended activation of PI3K and Akt/PKB but not Ras activation. Our findings suggest that MyoII and IQGAP/cortexillin play key roles in spatially and temporally regulating leading-edge activity and, through this, the ability of cells to restrict the site of pseudopod formation."", ""Cytokinesis in eukaryotic organisms is under the control of small GTP-binding proteins, although the underlying molecular mechanisms are not fully understood. Cortexillins are actin-binding proteins whose activity is crucial for cytokinesis in Dictyostelium. Here we show that the IQGAP-related and Rac1-binding protein DGAP1 specifically interacts with the C-terminal, actin-bundling doma

## 3. Wide per-query frame for Ex.1 (reranker comparison)

Compute rank-of-gold under each of BM25, MS-MARCO, BGE-m3, MedCPT, BGE-Gemma.
Then for each query also surface the **top-ranked non-gold article** under MS-MARCO
(the "competitor" we'll show in the table).

In [48]:
runs_ex1 = {
    "BM25": load_run(RUN_BM25_7A),
    "MS-MARCO": load_run(RUN_MSMARCO_7A),
    "BGE-m3": load_run(RUN_BGEM3_7A),
    "MedCPT": load_run(RUN_MEDCPT_7A),
    "Gemma": load_run(RUN_GEMMA_7A),
}

# Group each run by qid so we can do per-query lookups efficiently.
runs_ex1_by_qid = {
    name: {key[0]: g.drop("qid") for key, g in df.group_by("qid")}
    for name, df in runs_ex1.items()
}


def rank_of_gold_for_qid(name: str, qid: int, gold_pmids: list[str]) -> int | None:
    g = runs_ex1_by_qid[name].get(qid)
    if g is None:
        return None
    return rank_of_gold(g, gold_pmids)


def top_nongold(name: str, qid: int, gold_pmids: list[str], k: int = 1) -> list[str]:
    g = runs_ex1_by_qid[name].get(qid)
    if g is None:
        return []
    nongold = g.filter(~pl.col("docno").is_in(gold_pmids)).sort("rank").head(k)
    return nongold["docno"].to_list()


# Build wide frame.
rows = []
for r in gold.iter_rows(named=True):
    qid = r["qid"]
    gpmids = r["gold_pmids"]
    rec = {
        "qid": qid,
        "query_text": r["query_text"],
        "gold_pmids": gpmids,
        "gold_titles": r["gold_titles"],
    }
    for name in runs_ex1:
        rec[f"rank_{name}"] = rank_of_gold_for_qid(name, qid, gpmids)
    rec["top1_nongold_msmarco"] = (top_nongold("MS-MARCO", qid, gpmids, 1) or [None])[0]
    rec["top1_nongold_bm25"] = (top_nongold("BM25", qid, gpmids, 1) or [None])[0]
    rows.append(rec)

wide_ex1 = pl.DataFrame(rows)
print(wide_ex1.shape)
wide_ex1.head(3)

(1656, 11)


qid,query_text,gold_pmids,gold_titles,rank_BM25,rank_MS-MARCO,rank_BGE-m3,rank_MedCPT,rank_Gemma,top1_nongold_msmarco,top1_nongold_bm25
i64,str,list[str],list[str],i64,i64,i64,i64,i64,str,str
1,"""A basic region in the tail is …","[""24747353""]","[""The association of myosin IB with actin waves in dictyostelium requires both the plasma membrane-binding site and actin-binding region in the myosin tail.""]",1,5,3,4,2,"""18772133""","""18772133"""
10,"""A double mutant of gapA / rgaA…","[""20375144"", ""11447112""]","[""Involvement of the cytoskeleton in controlling leading-edge function during chemotaxis."", ""Recruitment of cortexillin into the cleavage furrow is controlled by Rac1 and IQGAP-related proteins.""]",1,3,1,2,1,"""9151691""","""9151691"""
100,"""Although the Dictyostelium MLC…","[""14570871""]","[""Regulatory mechanism of Dictyostelium myosin light chain kinase A.""]",1,3,2,1,1,"""1651931""","""8947030"""


## 4. Ex.1 candidates

Two examples to illustrate the discussion sentence:
> A reranker helps only when it preserves the gene/phenotype signal and adds useful
> discrimination; a weaker/general reranker can hurt.

We surface a handful of strong candidates for each direction. Selection is by inspection —
we look at the rank columns + query text + the "competitor" (top non-gold article under
the relevant reranker) and pick one whose mechanism is legible in a small table cell.

### Ex.1a — "Lexical signal lost"

Cases where BM25 ranks the gold high but MS-MARCO MiniLM demotes it severely.
Strong cases: `rank_BM25 ≤ 3` and `rank_MS-MARCO ≥ 50` and `rank_Gemma ≤ 3`
(so Gemma confirms the gold really is the right answer; the MS-MARCO drop isn't
just because the gold is borderline).

In [49]:
ex1a_pool = (
    wide_ex1
    .filter(
        pl.col("rank_BM25").is_not_null()
        & pl.col("rank_MS-MARCO").is_not_null()
        & pl.col("rank_Gemma").is_not_null()
    )
    # Moderate filter: BM25 ranks gold near top, MS-MARCO meaningfully demotes it,
    # Gemma still ranks it high (confirming gold is genuinely the right answer).
    .filter(pl.col("rank_BM25") <= 5)
    .filter(pl.col("rank_MS-MARCO") >= 20)
    .filter(pl.col("rank_Gemma") <= 5)
    .with_columns(
        (pl.col("rank_MS-MARCO") - pl.col("rank_BM25")).alias("msmarco_drop")
    )
    .sort("msmarco_drop", descending=True)
)
print(f"Ex.1a pool size: {len(ex1a_pool)}")
ex1a_pool.select(
    ["qid", "rank_BM25", "rank_MS-MARCO", "rank_BGE-m3", "rank_MedCPT", "rank_Gemma",
     "query_text", "top1_nongold_msmarco"]
).head(8)

Ex.1a pool size: 45


qid,rank_BM25,rank_MS-MARCO,rank_BGE-m3,rank_MedCPT,rank_Gemma,query_text,top1_nongold_msmarco
i64,i64,i64,i64,i64,i64,str,str
582,2,62,15,4,4,"""Rac GEF gxcT gxcT interacts wi…","""27313788"""
535,1,60,4,6,1,"""GefF has GEF activity towards …","""37492221"""
393,2,60,3,5,3,"""DymA also localizes to the pha…","""7820857"""
671,2,58,1,2,1,"""In addition, fszB null mutants…","""18990192"""
694,1,57,5,9,5,"""In addition, rasGEF GefF gefF …","""17380187"""
881,3,59,24,4,3,"""Mutants in which mybE is knock…","""8200481"""
1204,1,56,2,11,3,"""Strains carrying null mutation…","""14736886"""
1668,2,57,2,6,3,"""Upon chemotactic stimulation w…","""24608804"""


### Ex.1b — "Semantic rescue"

Cases where BM25 misses the gold (vocabulary gap) but a strong reranker bridges it.
Strong cases: `rank_BM25 ≥ 20` and `rank_Gemma ≤ 3`. We additionally require the
gold's title to share **few** content words with the query (proxy for vocabulary gap).

In [50]:
import re

_STOP = {
    "a","an","the","and","or","but","of","in","on","to","for","with","by","is","are",
    "was","were","be","been","being","that","this","these","those","as","at","from",
    "it","its","their","his","her","he","she","they","we","you","i","not","no","so",
    "if","then","than","also","into","via","such","using","via","can","may",
    "dictyostelium","discoideum",
}

def content_tokens(s: str) -> set[str]:
    toks = re.findall(r"[A-Za-z][A-Za-z0-9\-]+", s.lower())
    return {t for t in toks if t not in _STOP and len(t) > 2}

def lex_overlap_query_vs_titles(query: str, titles: list[str]) -> float:
    q = content_tokens(query)
    if not q:
        return 0.0
    best = 0.0
    for t in titles:
        tt = content_tokens(t)
        if not tt:
            continue
        ov = len(q & tt) / len(q)
        best = max(best, ov)
    return best


wide_ex1b = wide_ex1.with_columns(
    pl.struct(["query_text", "gold_titles"])
    .map_elements(lambda s: lex_overlap_query_vs_titles(s["query_text"], s["gold_titles"]),
                  return_dtype=pl.Float64)
    .alias("query_title_overlap")
)

ex1b_pool = (
    wide_ex1b
    .filter(
        pl.col("rank_BM25").is_not_null()
        & pl.col("rank_Gemma").is_not_null()
    )
    # Moderate filter: BM25 misses the gold (vocabulary gap), Gemma recovers it.
    .filter(pl.col("rank_BM25") >= 10)
    .filter(pl.col("rank_Gemma") <= 5)
    .filter(pl.col("query_title_overlap") <= 0.25)
    .with_columns(
        (pl.col("rank_BM25") - pl.col("rank_Gemma")).alias("gemma_lift")
    )
    .sort("gemma_lift", descending=True)
)
print(f"Ex.1b pool size: {len(ex1b_pool)}")
ex1b_pool.select(
    ["qid", "rank_BM25", "rank_MS-MARCO", "rank_BGE-m3", "rank_MedCPT", "rank_Gemma",
     "query_title_overlap", "query_text", "gold_titles", "top1_nongold_bm25"]
).head(8)

Ex.1b pool size: 127


qid,rank_BM25,rank_MS-MARCO,rank_BGE-m3,rank_MedCPT,rank_Gemma,query_title_overlap,query_text,gold_titles,top1_nongold_bm25
i64,i64,i64,i64,i64,i64,f64,str,list[str],str
1616,4529,6,5,11,2,0.0,"""Thissuggests that AlxA is some…","[""DdAlix, an Alix/AIP1 homolog in Dictyostelium discoideum, is required for multicellular development under low Ca2+ conditions.""]","""8394342"""
110,3081,83,81,6,4,0.0,"""An fhbA / fhbB double null mut…","[""Identification and characterization of two flavohemoglobin genes in Dictyostelium discoideum.""]","""18083829"""
1295,2279,76,17,1,2,0.0,"""The crlC- mutant has no detect…","[""A cAMP receptor-like G protein-coupled receptor with roles in growth regulation and development.""]","""11713669"""
540,1408,null,41,4,5,0.0,"""GefQ associates with F-actin, …","[""Linking Ras to myosin function: RasGEF Q, a Dictyostelium exchange factor for RasB, affects myosin II functions.""]","""17805484"""
807,858,8,4,15,3,0.1,"""Kif13 was found to co-purify a…","[""Identification of novel centrosomal proteins in Dictyostelium discoideum by comparative proteomic approaches.""]","""18712789"""
763,783,81,23,4,3,0.090909,"""In wild type cells, pks16 is e…","[""Role of fatty acid synthase in the development of Dictyostelium discoideum.""]","""41380999"""
690,546,1,1,1,1,0.181818,"""In addition, proteomic analysi…","[""Dictyostelium lipid droplets host novel proteins.""]","""38713739"""
553,529,2,5,5,3,0.086957,"""gerD encodes a 163 kDa protein…","[""A shared internal threonine-glutamic acid-threonine-proline repeat defines a family of Dictyostelium discoideum spore germination specific proteins.""]","""9334183"""


### Drill-down helper: show the query, the gold abstract, and the competitor abstract.

In [51]:
abstracts = pl.read_parquet(ABSTRACTS_PARQUET)
# Inspect columns to find the right names for pmid + title + abstract.
print(abstracts.columns)
abstracts.head(2)

['pmid', 'pmcid', 'doi', 'year', 'title', 'journal', 'authors', 'abstract_clean', 'file']


pmid,pmcid,doi,year,title,journal,authors,abstract_clean,file
str,str,str,str,str,str,str,str,str
"""2654141""","""PMC2115546""","""10.1083/jcb.108.5.1751""","""1989""","""Centrin-mediated microtubule s…","""The Journal of cell biology""","""Sanders MA, Salisbury JL.""","""Chlamydomonas cells excise the…","""article_fetching/output/all_cl…"
"""39528565""","""PMC11555045""","""10.1038/s41467-024-54272-4""","""2024""","""Nuclear localization sequence …","""Nature communications""","""Lim YJ, Yoon YJ, Lee H, Choi G…","""Plant pathogens secrete nuclea…","""article_fetching/output/all_cl…"


In [52]:
def lookup_article(pmid: str) -> dict:
    rec = abstracts.filter(pl.col("pmid").cast(pl.Utf8) == str(pmid))
    if rec.is_empty():
        return {"pmid": pmid, "title": None, "abstract": None}
    r = rec.row(0, named=True)
    title_col = "title" if "title" in r else next((c for c in r if "title" in c.lower()), None)
    abs_col = (
        "abstract_clean" if "abstract_clean" in r
        else next((c for c in r if "abstract" in c.lower()), None)
    )
    return {"pmid": pmid, "title": r.get(title_col), "abstract": r.get(abs_col)}


def drill_ex1(qid: int) -> None:
    row = wide_ex1.filter(pl.col("qid") == qid).row(0, named=True)
    print("=" * 80)
    print(f"qid {qid}  |  query: {row['query_text']}")
    print(
        f"  rank_BM25={row['rank_BM25']}  rank_MS-MARCO={row['rank_MS-MARCO']}  "
        f"rank_BGE-m3={row['rank_BGE-m3']}  rank_MedCPT={row['rank_MedCPT']}  "
        f"rank_Gemma={row['rank_Gemma']}"
    )
    for i, (pmid, title) in enumerate(zip(row["gold_pmids"], row["gold_titles"])):
        print(f"  GOLD #{i+1}  PMID {pmid}  |  {title}")
    for label, key in [("competitor (MS-MARCO top-1)", "top1_nongold_msmarco"),
                       ("competitor (BM25 top-1)", "top1_nongold_bm25")]:
        pmid = row.get(key)
        if pmid:
            art = lookup_article(pmid)
            print(f"  {label}  PMID {pmid}  |  {art['title']}")


# Example: drill into a few of the top candidates from each pool.
for qid in ex1a_pool.head(3)["qid"].to_list():
    drill_ex1(qid)
for qid in ex1b_pool.head(3)["qid"].to_list():
    drill_ex1(qid)

qid 582  |  query: Rac GEF gxcT gxcT interacts with and may also activate Rac1C.
  rank_BM25=2  rank_MS-MARCO=62  rank_BGE-m3=15  rank_MedCPT=4  rank_Gemma=4
  GOLD #1  PMID 24248334  |  Rho GTPases orient directional sensing in chemotaxis.
  competitor (MS-MARCO top-1)  PMID 27313788  |  ELMO1 Directly Interacts with Gβγ Subunit to Transduce GPCR Signaling to Rac1 Activation in Chemotaxis.
  competitor (BM25 top-1)  PMID 41511344  |  Phosphoproteomic Profiling Reveals Overlapping and Distinct Signaling Pathways in &lt;i&gt;Dictyostelium discoideum&lt;/i&gt; in Response to Two Different Chemorepellents.
qid 535  |  query: GefF has GEF activity towards RasG unpublished results, cited in.
  rank_BM25=1  rank_MS-MARCO=60  rank_BGE-m3=4  rank_MedCPT=6  rank_Gemma=1
  GOLD #1  PMID 30967009  |  Function of small GTPases in Dictyostelium macropinocytosis.
  competitor (MS-MARCO top-1)  PMID 37492221  |  Optogenetic modulation of guanine nucleotide exchange factors of Ras superfamily proteins

## 5. Ex.2 candidates — gene-aware query expansion

Build a small wide frame restricted to the QE benchmark (n=563): rank-of-gold under
BGE-m3 reranker when the query passed to the reranker is `body` / `+synonyms` / `+long`,
with the candidate pool fixed (this is the experimental setup behind Table S1).

We look for queries where +synonyms rescues the gold and where the added synonym appears
verbatim in the gold title or abstract — that makes the mechanism legible.

In [53]:
runs_ex2 = {
    "body": load_run(RUN_QE_BGEM3_BODY),
    "+syn": load_run(RUN_QE_BGEM3_SYN),
    "+long": load_run(RUN_QE_BGEM3_LONG),
}
runs_ex2_by_qid = {
    name: {key[0]: g.drop("qid") for key, g in df.group_by("qid")}
    for name, df in runs_ex2.items()
}

rows = []
for r in qe_bench.iter_rows(named=True):
    qid = r["qid"]
    gpmids = r["gold_pmids"]
    rec = {
        "qid": qid,
        "query_body": r["query_text_body"],
        "query_syn": r["query_text_synonyms"],
        "query_long": r["query_text_long"],
        "gold_pmids": gpmids,
        "gold_titles": r["gold_titles"],
        "gold_abstracts": r["gold_abstracts"],
        "detected_genes": r["detected_genes"],
    }
    for name in runs_ex2:
        g = runs_ex2_by_qid[name].get(qid)
        rec[f"rank_{name}"] = rank_of_gold(g, gpmids) if g is not None else None
    rows.append(rec)

wide_ex2 = pl.DataFrame(rows)
print(wide_ex2.shape)
wide_ex2.head(3)

(563, 11)


qid,query_body,query_syn,query_long,gold_pmids,gold_titles,gold_abstracts,detected_genes,rank_body,rank_+syn,rank_+long
i64,str,str,str,list[str],list[str],list[str],list[struct[4]],i64,i64,i64
100,"""Although the Dictyostelium MLC…","""Although the Dictyostelium MLC…","""Although the Dictyostelium MLC…","[""14570871""]","[""Regulatory mechanism of Dictyostelium myosin light chain kinase A.""]","[""In this study, we examined the activation mechanism of Dictyostelium myosin light chain kinase A (MLCK-A) using constitutively active Ca2+/calmodulin-dependent protein kinase kinase as a surrogate MLCK-A kinase. MLCK-A was phosphorylated at Thr166 by constitutively active Ca2+/calmodulin-dependent protein kinase kinase, resulting in an approximately 140-fold increase in catalytic activity, using intact Dictyostelium myosin II. Recombinant Dictyostelium myosin II regulatory light chain and Kemptamide were also readily phosphorylated by activated MLCK-A. Mass spectrometry analysis revealed that MLCK-A expressed by Escherichia coli was autophosphorylated at Thr289 and that, subsequent to Thr166 phosphorylation, MLCK-A also underwent a slow rate of autophosphorylation at multiple Ser residues. Using site-directed mutagenesis, we show that autophosphorylation at Thr289 is required for efficient phosphorylation and activation by an upstream kinase. By performing enzyme kinetics analysis on a series of MLCK-A truncation mutants, we found that residues 283-288 function as an autoinhibitory domain and that autoinhibition is fully relieved by Thr166 phosphorylation. Simple removal of this region resulted in a significant increase in the kcat of MLCK-A; however, it did not generate maximum enzymatic activity. Together with the results of our kinetic analysis of the enzymes, these findings demonstrate that Thr166 phosphorylation of MLCK-A by an upstream kinase subsequent to autophosphorylation at Thr289 results in generation of maximum MLCK-A activity through both release of an autoinhibitory domain from its catalytic core and a further increase (15-19-fold) in the kcat of the enzyme.""]","[{""DDB_G0279925"",""mlkA"",""MLCK, MLCK-A, MLCKA"",""myosin light chain kinase A""}]",2,2,2
1000,"""PIP3 negatively regulates Pten…","""PIP3 negatively regulates Pten…","""PIP3 negatively regulates Pten…","[""30367048""]","[""Mutual inhibition between PTEN and PIP3 generates bistability for polarity in motile cells.""]","[""Phosphatidylinositol 3,4,5-trisphosphate (PIP3) and PIP3 phosphatase (PTEN) are enriched mutually exclusively on the anterior and posterior membranes of eukaryotic motile cells. However, the mechanism that causes this spatial separation between the two molecules is unknown. Here we develop a method to manipulate PIP3 levels in living cells and used it to show PIP3 suppresses the membrane localization of PTEN. Single-molecule measurements of membrane-association and -dissociation kinetics and of lateral diffusion reveal that PIP3 suppresses the PTEN binding site required for stable PTEN membrane binding. Mutual inhibition between PIP3 and PTEN provides a mechanistic basis for bistability that creates a PIP3-enriched/PTEN-excluded state and a PTEN-enriched/PIP3-excluded state underlying the strict spatial separation between PIP3 and PTEN. The PTEN binding site also mediates the suppression of PTEN membrane localization in chemotactic signaling. These results illustrate that the PIP3-PTEN bistable system underlies a cell's decision-making for directional movement irrespective of the environment.""]","[{""DDB_G0286557"",""pten"",""ptenA, protein tyrosine phosphatase, 3-phosphatidylinositol 3-phosphatase"",""phosphatidylinositol 3,4,5-trisphosphate 3-phosphatase""}]",1,1,1
1001,"""PIP5 kinase, PikI, produces PI…","""PIP5 kinase, PikI, produces PI…","""PIP5 kinase, PikI, produces PI…","[""24485835""]","[""A PIP5 kinase essential for efficient chemotactic signaling.""]","[""In neutrophils and Dictyostelium, chemoattractant gradients generate dir

In [54]:
def added_terms(orig: str, expanded: str | None) -> str:
    """Crude diff: tokens in `expanded` that are not in `orig`, preserving order."""
    if not expanded:
        return ""
    orig_toks = set(re.findall(r"[A-Za-z][A-Za-z0-9\-]+", orig.lower()))
    out, seen = [], set()
    for tok in re.findall(r"[A-Za-z][A-Za-z0-9\-]+", expanded):
        if tok.lower() in orig_toks:
            continue
        if tok.lower() in seen:
            continue
        seen.add(tok.lower())
        out.append(tok)
    return " ".join(out)


def added_term_in_gold(added: str, titles: list[str], abstracts: list[str]) -> bool:
    if not added:
        return False
    text = " ".join(titles + abstracts).lower()
    for tok in added.split():
        if tok.lower() in text:
            return True
    return False


wide_ex2 = wide_ex2.with_columns([
    pl.struct(["query_body", "query_syn"])
    .map_elements(lambda s: added_terms(s["query_body"], s["query_syn"]),
                  return_dtype=pl.Utf8)
    .alias("added_syn"),
    pl.struct(["query_body", "query_long"])
    .map_elements(lambda s: added_terms(s["query_body"], s["query_long"]),
                  return_dtype=pl.Utf8)
    .alias("added_long"),
])
wide_ex2 = wide_ex2.with_columns(
    pl.struct(["added_syn", "gold_titles", "gold_abstracts"])
    .map_elements(lambda s: added_term_in_gold(s["added_syn"], s["gold_titles"], s["gold_abstracts"]),
                  return_dtype=pl.Boolean)
    .alias("syn_in_gold")
)

ex2_pool = (
    wide_ex2
    .filter(pl.col("rank_body").is_not_null() & pl.col("rank_+syn").is_not_null())
    .filter(pl.col("rank_body") >= 10)
    .filter(pl.col("rank_+syn") <= 3)
    .filter(pl.col("syn_in_gold"))
    .with_columns(
        (pl.col("rank_body") - pl.col("rank_+syn")).alias("syn_lift")
    )
    .sort("syn_lift", descending=True)
)
print(f"Ex.2 pool size: {len(ex2_pool)}")
ex2_pool.select(
    ["qid", "rank_body", "rank_+syn", "rank_+long",
     "added_syn", "query_body", "gold_titles"]
).head(8)

Ex.2 pool size: 19


qid,rank_body,rank_+syn,rank_+long,added_syn,query_body,gold_titles
i64,i64,i64,i64,str,str,list[str]
460,565,3,5,"""DdFHa DidiA""","""fhbA is expressed in growing c…","[""Identification and characterization of two flavohemoglobin genes in Dictyostelium discoideum.""]"
441,255,3,4,"""RasGEFS""","""Expression of gefS during grow…","[""The Dictyostelium genome encodes numerous RasGEFs with multiple biological roles.""]"
279,167,1,9,"""DP87 SP75""","""cotD mRNA accumulates slightly…","[""Developmental regulation of transcription of a novel prespore-specific gene (Dp87) in Dictyostelium discoideum.""]"
1166,73,1,1,"""alg2A DdPEF-1""","""show that pefA expressionis lo…","[""Identification and characterization of two penta-EF-hand Ca(2+)-binding proteins in Dictyostelium discoideum.""]"
313,39,1,1,"""DET1""","""DetA is involved in cell type …","[""Characterization of the Dictyostelium homolog of chromatin binding protein DET1 suggests a conserved pathway regulating cell type specification and developmental plasticity.""]"
299,29,3,3,"""PI5P4 RhoGAP domain-containing…","""Dd5P4 null mutants exhibit slo…","[""A diverse family of inositol 5-phosphatases playing a role in growth and development in Dictyostelium discoideum.""]"
529,29,3,5,"""RasGEFB""","""gefB expression also increases…","[""Gamete fusion and cytokinesis preceding zygote establishment in the sexual process of Dictyostelium discoideum.""]"
1442,26,1,2,"""S-adenosyl-L-homocysteine hydr…","""The sahA gene was later identi…","[""Cloning of a cDNA for the S-adenosyl-L-homocysteine hydrolase from Dictyostelium discoideum."", ""Amino acid sequence of S-adenosyl-L-homocysteine hydrolase from Dictyostelium discoideum as deduced from the cDNA sequence.""]"


## 6. Ex.3 candidates — full-text chunks help abstract-insufficient claims

Restrict to queries whose gold doc is labeled `abstract_insufficient`, then find ones
where the rank improves from "abstract-only BGE-m3" to "+chunks BGE-m3".

For each candidate we also need a body-chunk excerpt that contains the cue missing from
the abstract. The helper below pulls the top-ranked chunks for the gold PMID from the
chunked corpus.

In [55]:
runs_ex3 = {
    "abs": load_run(RUN_ABS_BGEM3_7A),
    "chunks": load_run(RUN_CHUNK_BGEM3_7A),
}
runs_ex3_by_qid = {
    name: {key[0]: g.drop("qid") for key, g in df.group_by("qid")}
    for name, df in runs_ex3.items()
}


def insufficient_gold_pmids(r: dict) -> list[str]:
    return [
        str(p)
        for p, lvl in zip(r["gold_pmids"], r["evidence_levels"])
        if lvl == "abstract_insufficient"
    ]


rows = []
for r in gold.iter_rows(named=True):
    insuff = insufficient_gold_pmids(r)
    if not insuff:
        continue
    qid = r["qid"]
    rec = {
        "qid": qid,
        "query_text": r["query_text"],
        "gold_pmids": r["gold_pmids"],
        "gold_titles": r["gold_titles"],
        "gold_abstracts": r["gold_abstracts"],
        "insufficient_pmids": insuff,
    }
    g_abs = runs_ex3_by_qid["abs"].get(qid)
    g_chk = runs_ex3_by_qid["chunks"].get(qid)
    # For the chunked corpus, doc IDs include chunk suffixes like "PMID#body_001";
    # collapse to PMID first-occurrence.
    if g_chk is not None:
        g_chk = (
            g_chk.with_columns(pl.col("docno").str.split("#").list.first().alias("pmid"))
            .sort("rank")
            .group_by("pmid", maintain_order=True)
            .agg(pl.col("rank").min().alias("rank"))
            .rename({"pmid": "docno"})
        )
    rec["rank_abs_any_gold"] = rank_of_gold(g_abs, r["gold_pmids"]) if g_abs is not None else None
    rec["rank_chk_any_gold"] = rank_of_gold(g_chk, r["gold_pmids"]) if g_chk is not None else None
    rec["rank_abs_insuff"] = rank_of_gold(g_abs, insuff) if g_abs is not None else None
    rec["rank_chk_insuff"] = rank_of_gold(g_chk, insuff) if g_chk is not None else None
    rows.append(rec)

wide_ex3 = pl.DataFrame(rows)
print(wide_ex3.shape)

ex3_pool = (
    wide_ex3
    .filter(
        pl.col("rank_abs_insuff").is_not_null()
        & pl.col("rank_chk_insuff").is_not_null()
    )
    .filter(pl.col("rank_abs_insuff") >= 50)
    .filter(pl.col("rank_chk_insuff") <= 10)
    # Single insufficient gold PMID makes the example unambiguous.
    .filter(pl.col("insufficient_pmids").list.len() == 1)
    .with_columns(
        (pl.col("rank_abs_insuff") - pl.col("rank_chk_insuff")).alias("chunk_lift")
    )
    .sort("chunk_lift", descending=True)
)
print(f"Ex.3 pool size: {len(ex3_pool)}")
ex3_pool.select(
    ["qid", "rank_abs_insuff", "rank_chk_insuff", "query_text", "gold_titles",
     "insufficient_pmids"]
).head(8)

(387, 10)
Ex.3 pool size: 8


qid,rank_abs_insuff,rank_chk_insuff,query_text,gold_titles,insufficient_pmids
i64,i64,i64,str,list[str],list[str]
840,80,3,"""Mammalian RHEB is inactivated …","[""TOR complex 2 (TORC2) in Dictyostelium suppresses phagocytic nutrient capture independently of TORC1-mediated nutrient sensing.""]","[""22266904""]"
1342,63,1,"""The histone H1 linker protein …","[""Molecular cloning of a cDNA encoding the nucleosome core histone H3 from Dictyostelium discoideum by genetic screening in yeast.""]","[""9177486""]"
584,65,3,"""H3b and H3c encode proteins 13…","[""Molecular cloning of a cDNA encoding the nucleosome core histone H3 from Dictyostelium discoideum by genetic screening in yeast.""]","[""9177486""]"
534,54,1,"""gefE null mutants are also les…","[""Developmental lineage priming in Dictyostelium by heterogeneous Ras activation.""]","[""24282234""]"
303,54,3,"""Deletion of all 3 csb genes de…","[""Cell-cell adhesion prevents mutant cells lacking myosin II from penetrating aggregation streams of Dictyostelium.""]","[""8626027""]"
119,52,2,"""Annexin VII interacts with the…","[""The annexins of Dictyostelium.""]","[""16762449""]"
376,59,10,"""DRG xacA contains both GAP and…","[""The Dictyostelium Bcr/Abr-related protein DRG regulates both Rac- and Rab-dependent pathways."", ""Cloning and characterization of a rhoGAP homolog from Dictyostelium discoideum.""]","[""9188459""]"
384,51,2,"""During development, expression…","[""A protein containing a serine-rich domain with vesicle fusing properties mediates cell cycle-dependent cytosolic pH regulation.""]","[""10747962""]"


### Body-chunk lookup

For a given (qid, gold pmid), surface the top-ranked body chunks for that PMID under the
chunked-corpus run, then pull their text from `pdf_extraction/v2/chunks.jsonl`.

In [56]:
# Build a chunk-text index once. Indexed by chunk_id.
chunk_text: dict[str, dict] = {}
with open(CHUNKS_JSONL) as fh:
    for line in fh:
        c = json.loads(line)
        chunk_text[c["chunk_id"]] = {
            "pmid": str(c["pmid"]),
            "type": c.get("type"),
            "seq": c.get("seq"),
            "text": c["text"],
        }
print(f"chunks indexed: {len(chunk_text)}")

# Re-load the chunked rerank run preserving the chunk-level docnos so we can find which
# chunks scored highest.
chunked_run_full = load_run(RUN_CHUNK_BGEM3_7A)
chunked_run_full = chunked_run_full.with_columns(
    pl.col("docno").str.split("#").list.first().alias("pmid")
)


def top_chunks_for_pmid(qid: int, pmid: str, k: int = 3) -> list[dict]:
    rows = chunked_run_full.filter(
        (pl.col("qid") == qid) & (pl.col("pmid") == str(pmid))
    ).sort("rank").head(k)
    out = []
    for r in rows.iter_rows(named=True):
        info = chunk_text.get(r["docno"], {})
        out.append({"rank": r["rank"], "chunk_id": r["docno"], **info})
    return out


def drill_ex3(qid: int) -> None:
    row = wide_ex3.filter(pl.col("qid") == qid).row(0, named=True)
    print("=" * 80)
    print(f"qid {qid}  |  query: {row['query_text']}")
    print(
        f"  rank_abs (insuff)={row['rank_abs_insuff']}   "
        f"rank_chunks (insuff)={row['rank_chk_insuff']}"
    )
    for pmid in row["insufficient_pmids"]:
        idx = row["gold_pmids"].index(pmid)
        title = row["gold_titles"][idx]
        abstract = row["gold_abstracts"][idx]
        print(f"  GOLD PMID {pmid}  |  {title}")
        print(f"  ABSTRACT: {abstract[:400]}...")
        for ch in top_chunks_for_pmid(qid, pmid, k=3):
            txt = (ch.get("text") or "")[:400].replace("\n", " ")
            print(f"  chunk rank={ch['rank']}  {ch['chunk_id']}  ({ch.get('type')}): {txt}...")


for qid in ex3_pool.head(3)["qid"].to_list():
    drill_ex3(qid)

chunks indexed: 87028
qid 840  |  query: Mammalian RHEB is inactivated by TSC2, suggesting that Dictyostelium Rheb is inactivated by RasGAP tsc2 Huang, et al 2008, PMID:18411301.
  rank_abs (insuff)=80   rank_chunks (insuff)=3
  GOLD PMID 22266904  |  TOR complex 2 (TORC2) in Dictyostelium suppresses phagocytic nutrient capture independently of TORC1-mediated nutrient sensing.
  ABSTRACT: The TOR protein kinase functions in two distinct complexes, TOR complex 1 (TORC1) and 2 (TORC2). TORC1 is required for growth in response to growth factors, nutrients and the cellular energy state; TORC2 regulates AKT signaling, which can modulate cytoskeletal polarization. In its ecological niche, Dictyostelium engulf bacteria and yeast for nutrient capture. Despite the essential role of TORC1 in...
  chunk rank=3  22266904#body_030  (body): es pombe and mammalian cells indicate a potential function for Rheb that is independent of TORC1 (Aspuria et al., 2007; Karbowniczek et al., 2004; Otsubo and Yam

## 7. Sentence-excerpt helpers

To make each example legible in a compact table, we surface **one short sentence**
from the abstract/body — the one whose content tokens best overlap with the query.
Sentences are trimmed to ~180 characters with an ellipsis.

In [57]:
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z(])")


def split_sentences(text: str) -> list[str]:
    if not text:
        return []
    # Light cleanup; strip section-header artifacts and excessive whitespace.
    t = re.sub(r"\s+", " ", text).strip()
    return [s.strip() for s in _SENT_SPLIT.split(t) if s.strip()]


def best_sentence(
    query: str, text: str, must_include: list[str] | None = None, max_chars: int = 180
) -> str:
    """Pick the sentence whose content-token overlap with `query` is highest.

    If `must_include` is given, restrict to sentences containing any of those tokens
    (case-insensitive substring), then pick the best by query overlap.
    Returns truncated string with ellipsis.
    """
    sents = split_sentences(text)
    if not sents:
        return ""
    qtoks = content_tokens(query)
    if must_include:
        needles = [n.lower() for n in must_include if n]
        filtered = [
            s for s in sents
            if any(n in s.lower() for n in needles)
        ]
        if filtered:
            sents = filtered
    scored = []
    for s in sents:
        st = content_tokens(s)
        ov = len(qtoks & st) / max(len(qtoks), 1)
        scored.append((ov, len(s), s))
    # Highest overlap; tie-break by shorter sentence (more focused).
    scored.sort(key=lambda x: (-x[0], x[1]))
    best = scored[0][2]
    return shorten(best, max_chars)


def shorten(text: str, max_chars: int) -> str:
    text = text.strip()
    if len(text) <= max_chars:
        return text
    cut = text[:max_chars].rsplit(" ", 1)[0]
    return cut + "..."


def short_title(title: str, max_chars: int = 75) -> str:
    return shorten(title or "", max_chars)

### Unified candidate inspector

Prints the top-N candidates from each pool with full mechanism context (query,
gold title + best sentence, competitor title + best sentence, ranks). Use this
to pick the SELECTED_* qids.

In [58]:
def show_ex1_candidates(pool: pl.DataFrame, label: str, n: int = 5) -> None:
    print(f"\n{'=' * 80}\nEx.1 — {label}  (top {n} of {len(pool)})\n{'=' * 80}")
    for r in pool.head(n).iter_rows(named=True):
        qid = r["qid"]
        gold_pmid = r["gold_pmids"][0]
        gold_title = r["gold_titles"][0]
        gold_abs = lookup_article(gold_pmid)["abstract"] or ""
        gold_sent = best_sentence(r["query_text"], gold_abs)
        comp_key = "top1_nongold_msmarco" if "lost" in label.lower() else "top1_nongold_bm25"
        comp_pmid = r[comp_key]
        comp = lookup_article(comp_pmid) if comp_pmid else None
        comp_sent = best_sentence(r["query_text"], comp["abstract"] or "") if comp else ""
        print(f"\nqid={qid}  BM25={r['rank_BM25']}  MS-MARCO={r['rank_MS-MARCO']}  "
              f"BGE-m3={r['rank_BGE-m3']}  MedCPT={r['rank_MedCPT']}  Gemma={r['rank_Gemma']}")
        print(f"  query : {r['query_text']}")
        print(f"  GOLD  : [{gold_pmid}] {short_title(gold_title, 90)}")
        print(f"        : \"{gold_sent}\"")
        if comp:
            print(f"  COMP  : [{comp_pmid}] {short_title(comp['title'], 90)}")
            print(f"        : \"{comp_sent}\"")


def show_ex2_candidates(pool: pl.DataFrame, n: int = 5) -> None:
    print(f"\n{'=' * 80}\nEx.2 — Query expansion  (top {n} of {len(pool)})\n{'=' * 80}")
    for r in pool.head(n).iter_rows(named=True):
        qid = r["qid"]
        gold_pmid = r["gold_pmids"][0]
        gold_title = r["gold_titles"][0]
        gold_abs = r["gold_abstracts"][0] or ""
        needles = r["added_syn"].split() if r["added_syn"] else None
        gold_sent = best_sentence(r["query_body"], gold_abs, must_include=needles)
        print(f"\nqid={qid}  body={r['rank_body']}  +syn={r['rank_+syn']}  "
              f"+long={r['rank_+long']}")
        print(f"  original  : {r['query_body']}")
        print(f"  + syn adds: {r['added_syn']}")
        print(f"  GOLD      : [{gold_pmid}] {short_title(gold_title, 90)}")
        print(f"            : \"{gold_sent}\"")


def show_ex3_candidates(pool: pl.DataFrame, n: int = 5) -> None:
    print(f"\n{'=' * 80}\nEx.3 — Full-text  (top {n} of {len(pool)})\n{'=' * 80}")
    for r in pool.head(n).iter_rows(named=True):
        qid = r["qid"]
        pmid = r["insufficient_pmids"][0]
        idx = r["gold_pmids"].index(pmid)
        title = r["gold_titles"][idx]
        abstract = r["gold_abstracts"][idx] or ""
        abs_sent = best_sentence(r["query_text"], abstract)
        body_text = ""
        for ch in top_chunks_for_pmid(qid, pmid, k=5):
            if ch.get("type") == "body" and ch.get("text"):
                body_text = ch["text"]
                break
        body_sent = best_sentence(r["query_text"], body_text)
        print(f"\nqid={qid}  abs={r['rank_abs_insuff']}  +chunks={r['rank_chk_insuff']}")
        print(f"  claim : {r['query_text']}")
        print(f"  GOLD  : [{pmid}] {short_title(title, 90)}")
        print(f"  abs   : \"{abs_sent}\"")
        print(f"  body  : \"{body_sent}\"")


def drill_ex2(qid: int) -> None:
    row = wide_ex2.filter(pl.col("qid") == qid).row(0, named=True)
    print("=" * 80)
    print(f"qid {qid}")
    print(f"  ORIGINAL: {row['query_body']}")
    print(f"  + SYN ADDS: {row['added_syn']}")
    print(
        f"  rank_body={row['rank_body']}  rank_+syn={row['rank_+syn']}  "
        f"rank_+long={row['rank_+long']}"
    )
    for pmid, title, abstract in zip(row["gold_pmids"], row["gold_titles"], row["gold_abstracts"]):
        print(f"  GOLD PMID {pmid}  |  {title}")
        needles = row["added_syn"].split() if row["added_syn"] else None
        s = best_sentence(row["query_body"], abstract or "", must_include=needles)
        print(f"    sentence-with-synonym: \"{s}\"")

## 8. Manuscript-ready preview blocks

Set the four `SELECTED_*` qids below, then run the next cell. Each example emits:
1. A markdown preview (easy to scan in the notebook), and
2. A LaTeX `tabular` snippet ready to paste into `paper.tex`.

Ex.1 uses only three reranker columns (BM25 / MS-MARCO / Gemma) to keep the table
narrow; BGE-m3 and MedCPT ranks are listed in the markdown header for reference.

In [59]:
SELECTED_EX1A: int | None = 393    # DymA / cleavage furrow — competitor (alpha-actinin paper) has matching phenotype location words but no DymA
SELECTED_EX1B: int | None = 1616   # AlxA <-> DdAlix — same gene, alternate name; BM25 cannot bridge
SELECTED_EX2: int | None = 313     # DetA -> DET1 — single canonical synonym appears in gold title
SELECTED_EX3: int | None = 534     # gefE / DIF sensitivity — body sentence near-paraphrases the claim

# Brainstorm: also render qid 535 (GefF / RasG) as an alternative Ex.1a whose competitor
# is itself a Ras-GEF paper (same gene family) — a more "biologically plausible mistake"
# than the DymA case (where the competitor is topically Dicty but a different protein class).
ALT_EX1A: int | None = 535

# Optional cleanup of curator-text artifacts in displayed queries (renderers only;
# the underlying retrieval ran on the original strings). Truncations use [...].
QUERY_OVERRIDES: dict[int, str] = {
    535: "GefF has GEF activity towards RasG [...]",  # original trailed: "unpublished results, cited in."
}

# Mechanism annotations rendered as a "Why" line under each example block.
WHY_NOTES: dict[int, str] = {
    393: (
        "MS-MARCO promotes a topically-related Dicty paper that shares the claim's "
        "phenotype-location vocabulary (cleavage furrow, phagocytic cup) but does not "
        "study DymA."
    ),
    535: (
        "MS-MARCO promotes a paper about Ras-superfamily GEFs (same gene family as the "
        "gold) but loses the specific GefF--RasG link the curator cited."
    ),
    1616: (
        "BM25 cannot bridge AlxA (curator's gene symbol) and DdAlix (the literature "
        "name for the same gene); Gemma matches them on semantic similarity."
    ),
}


def _ex1_competitor(row: dict, label: str) -> str | None:
    """For 'lost' we show what MS-MARCO promoted; for 'rescue' what BM25 promoted."""
    key = "top1_nongold_msmarco" if "lost" in label.lower() else "top1_nongold_bm25"
    return row.get(key)


def _display_query(qid: int, original: str) -> str:
    return QUERY_OVERRIDES.get(qid, original)


def render_ex1_markdown(qid: int, label: str) -> str:
    row = wide_ex1.filter(pl.col("qid") == qid).row(0, named=True)
    query_disp = _display_query(qid, row["query_text"])
    gold_pmid = row["gold_pmids"][0]
    gold_title = row["gold_titles"][0]
    gold_abs = lookup_article(gold_pmid)["abstract"] or ""
    gold_sent = best_sentence(row["query_text"], gold_abs)

    comp_pmid = _ex1_competitor(row, label)
    comp = lookup_article(comp_pmid) if comp_pmid else None
    comp_sent = best_sentence(row["query_text"], comp["abstract"] or "") if comp else ""
    c_ranks = (
        {n: rank_of_gold_for_qid(n, qid, [comp_pmid]) for n in runs_ex1}
        if comp_pmid else {}
    )

    why = WHY_NOTES.get(qid)
    # Two-column box: left = role (Query / Gold / Competitor / Why), right = content.
    # Title + snippet stacked inside the right cell via <br>.
    lines = [
        f"### Ex.1 — {label} &nbsp;(qid {qid})",
        "",
        "| | |",
        "|---|---|",
        f"| **Query** | {query_disp} |",
        (
            f"| **Gold** | PMID {gold_pmid} &nbsp; *{short_title(gold_title, 95)}*"
            f"<br>&nbsp;&nbsp;&nbsp;&nbsp;\"{gold_sent}\""
            f"<br>&nbsp;&nbsp;&nbsp;&nbsp;ranks: BM25 **{row['rank_BM25']}**, "
            f"MS-MARCO **{row['rank_MS-MARCO']}**, Gemma **{row['rank_Gemma']}** "
            f"(BGE-m3 {row['rank_BGE-m3']}, MedCPT {row['rank_MedCPT']}) |"
        ),
    ]
    if comp:
        lines.append(
            f"| **Competitor** | PMID {comp_pmid} &nbsp; *{short_title(comp['title'], 95)}*"
            f"<br>&nbsp;&nbsp;&nbsp;&nbsp;\"{comp_sent}\""
            f"<br>&nbsp;&nbsp;&nbsp;&nbsp;ranks: BM25 **{c_ranks['BM25']}**, "
            f"MS-MARCO **{c_ranks['MS-MARCO']}**, Gemma **{c_ranks['Gemma']}** |"
        )
    if why:
        lines.append(f"| **Why** | _{why}_ |")
    return "\n".join(lines) + "\n"


def _esc(s: str) -> str:
    """Minimal LaTeX escape for fields we paste into tabulars."""
    if s is None:
        return ""
    return (
        s.replace("\\", "\\textbackslash{}")
         .replace("&", "\\&")
         .replace("%", "\\%")
         .replace("$", "\\$")
         .replace("#", "\\#")
         .replace("_", "\\_")
         .replace("{", "\\{")
         .replace("}", "\\}")
         .replace("~", "\\textasciitilde{}")
         .replace("^", "\\textasciicircum{}")
    )


def render_ex1_latex_block(qid: int, label: str) -> str:
    """Returns the inner rows for one example block (no \\begin{tabular})."""
    row = wide_ex1.filter(pl.col("qid") == qid).row(0, named=True)
    query_disp = _display_query(qid, row["query_text"])
    gold_pmid = row["gold_pmids"][0]
    gold_title = short_title(row["gold_titles"][0])
    gold_abs = lookup_article(gold_pmid)["abstract"] or ""
    gold_sent = best_sentence(row["query_text"], gold_abs)

    comp_pmid = _ex1_competitor(row, label)
    comp = lookup_article(comp_pmid) if comp_pmid else None
    comp_sent = best_sentence(row["query_text"], comp["abstract"] or "") if comp else ""
    c_ranks = (
        {n: rank_of_gold_for_qid(n, qid, [comp_pmid]) for n in runs_ex1}
        if comp_pmid else {}
    )
    comp_title = short_title(comp["title"]) if comp else ""

    lines = [
        f"\\multicolumn{{4}}{{@{{}}l@{{}}}}{{\\emph{{{_esc(label)}.}} "
        f"\\textit{{Query:}} {_esc(query_disp)}}} \\\\",
        f"Gold PMID {gold_pmid} --- {_esc(gold_title)} & "
        f"{row['rank_BM25']} & {row['rank_MS-MARCO']} & {row['rank_Gemma']} \\\\",
        f"\\multicolumn{{4}}{{@{{}}l@{{}}}}{{\\quad\\small\\itshape "
        f"``{_esc(gold_sent)}''}} \\\\",
    ]
    if comp:
        lines += [
            f"Competitor PMID {comp_pmid} --- {_esc(comp_title)} & "
            f"{c_ranks['BM25']} & {c_ranks['MS-MARCO']} & {c_ranks['Gemma']} \\\\",
            f"\\multicolumn{{4}}{{@{{}}l@{{}}}}{{\\quad\\small\\itshape "
            f"``{_esc(comp_sent)}''}} \\\\",
        ]
    why = WHY_NOTES.get(qid)
    if why:
        lines.append(
            f"\\multicolumn{{4}}{{@{{}}p{{\\linewidth}}@{{}}}}{{\\quad\\footnotesize "
            f"\\textit{{Why:}} {_esc(why)}}} \\\\"
        )
    return "\n".join(lines)


def render_ex1_latex(qids_labels: list[tuple[int, str]]) -> str:
    blocks = []
    for i, (qid, label) in enumerate(qids_labels):
        if i:
            blocks.append("\\midrule")
        blocks.append(render_ex1_latex_block(qid, label))
    body = "\n".join(blocks)
    return (
        "\\begin{table}[t]\n"
        "\\centering\n"
        "\\caption{Qualitative examples illustrating reranker behavior on two curator-claim "
        "queries. Each block shows the gold article and one competitor article "
        "(the top-ranked non-gold article under the weaker reranker for 'Lexical signal lost', "
        "or under BM25 for 'Semantic rescue'). Ranks are positions of each article in the "
        "final ranking; lower is better. Quoted lines are best-matching sentences from "
        "each article's abstract.}\n"
        "\\label{tab:rerank_examples}\n"
        "\\scriptsize\n"
        "\\setlength{\\tabcolsep}{4pt}\n"
        "\\begin{tabular}{@{}p{8.0cm}rrr@{}}\n"
        "\\toprule\n"
        "Article & BM25 & MS-MARCO & Gemma \\\\\n"
        "\\midrule\n"
        f"{body}\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\end{table}"
    )


def _gene_mapping(detected_genes: list[dict]) -> str:
    """Format detected gene(s) as 'curator-mention -> canonical (synonyms)'.

    The dictyBase pipeline stores `gene_name` as the canonical symbol and `synonyms`
    as the comma-separated list of alternative spellings; the curator typically uses
    one of the synonyms in the claim and the +syn expansion appends the canonical.
    """
    if not detected_genes:
        return ""
    parts = []
    for g in detected_genes:
        canonical = g.get("gene_name") or ""
        syns = g.get("synonyms") or ""
        if syns:
            parts.append(f"`{canonical}` (synonyms: {syns})")
        else:
            parts.append(f"`{canonical}`")
    return "; ".join(parts)


def render_ex2_markdown(qid: int) -> str:
    row = wide_ex2.filter(pl.col("qid") == qid).row(0, named=True)
    gold_pmid = row["gold_pmids"][0]
    gold_title = row["gold_titles"][0]
    gold_abs = row["gold_abstracts"][0] or ""
    needles = row["added_syn"].split() if row["added_syn"] else None
    gold_sent = best_sentence(row["query_body"], gold_abs, must_include=needles)
    gene_str = _gene_mapping(row.get("detected_genes") or [])
    # Extract the exact line appended to the query (everything after the original body).
    appended = ""
    if row["query_syn"] and row["query_syn"].startswith(row["query_body"]):
        appended = row["query_syn"][len(row["query_body"]):].strip()
    return (
        f"### Ex.2 — Gene-aware query expansion &nbsp;(qid {qid})\n\n"
        f"| | |\n"
        f"|---|---|\n"
        f"| **Original query** | {row['query_body']} |\n"
        f"| **Detected gene** | {gene_str} |\n"
        f"| **Appended to query** | `{appended}` &nbsp;_(synonym `{row['added_syn']}` "
        f"attached to curator mention)_ |\n"
        f"| **Gold** | PMID {gold_pmid} &nbsp; *{short_title(gold_title, 100)}*"
        f"<br>&nbsp;&nbsp;&nbsp;&nbsp;\"{gold_sent}\" |\n"
        f"| **Rank of gold** | body **{row['rank_body']}** &nbsp;→&nbsp; "
        f"+syn **{row['rank_+syn']}** &nbsp;→&nbsp; "
        f"+syn\\&prod **{row['rank_+long']}** |\n"
    )


def render_ex2_latex(qid: int) -> str:
    row = wide_ex2.filter(pl.col("qid") == qid).row(0, named=True)
    gold_pmid = row["gold_pmids"][0]
    gold_title = short_title(row["gold_titles"][0], 95)
    gold_abs = row["gold_abstracts"][0] or ""
    needles = row["added_syn"].split() if row["added_syn"] else None
    gold_sent = best_sentence(row["query_body"], gold_abs, must_include=needles)
    appended = ""
    if row["query_syn"] and row["query_syn"].startswith(row["query_body"]):
        appended = row["query_syn"][len(row["query_body"]):].strip()
    # Format detected gene briefly: "curator-mention -> canonical".
    genes = row.get("detected_genes") or []
    gene_brief = ""
    if genes:
        g = genes[0]
        gene_brief = f"\\texttt{{{_esc(g.get('gene_name') or '')}}} (synonyms: " \
                     f"{_esc(g.get('synonyms') or '')})"
    return (
        "\\begin{table}[t]\n"
        "\\centering\n"
        "\\caption{Qualitative example illustrating gene-aware query expansion. "
        "The curator's claim mentions a gene by one of its alternative names; the "
        "expansion appends the canonical name (and optionally a product description) "
        "drawn from dictyBase metadata for the detected gene. Ranks are positions in "
        "the BGE-reranker-v2-m3 output on a fixed candidate pool.}\n"
        "\\label{tab:qe_example}\n"
        "\\scriptsize\n"
        "\\setlength{\\tabcolsep}{6pt}\n"
        "\\begin{tabular}{@{}p{3.6cm}p{7.4cm}@{}}\n"
        "\\toprule\n"
        f"Original query & {_esc(row['query_body'])} \\\\\n"
        f"Detected gene & {gene_brief} \\\\\n"
        f"Appended to query & \\texttt{{{_esc(appended)}}} \\\\\n"
        f"Gold (PMID {gold_pmid}) & \\emph{{{_esc(gold_title)}}} \\\\\n"
        f"Gold sentence with synonym & \\small\\itshape ``{_esc(gold_sent)}'' \\\\\n"
        "\\midrule\n"
        "\\multicolumn{2}{@{}l@{}}{Rank of gold: "
        f"body = {row['rank_body']}, "
        f"+ synonyms = {row['rank_+syn']}, "
        f"+ syn \\& prod = {row['rank_+long']}.}} \\\\\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\end{table}"
    )


def render_ex3_markdown(qid: int) -> str:
    row = wide_ex3.filter(pl.col("qid") == qid).row(0, named=True)
    pmid = row["insufficient_pmids"][0]
    idx = row["gold_pmids"].index(pmid)
    title = row["gold_titles"][idx]
    abstract = row["gold_abstracts"][idx] or ""
    abs_sent = best_sentence(row["query_text"], abstract)
    body_text = ""
    for ch in top_chunks_for_pmid(qid, pmid, k=5):
        if ch.get("type") == "body" and ch.get("text"):
            body_text = ch["text"]
            break
    body_sent = best_sentence(row["query_text"], body_text)
    return (
        f"### Ex.3 — Full-text rescues abstract-insufficient claim &nbsp;(qid {qid})\n\n"
        f"| | |\n"
        f"|---|---|\n"
        f"| **Curator claim** | {row['query_text']} |\n"
        f"| **Gold** | PMID {pmid} &nbsp; *{short_title(title, 100)}* |\n"
        f"| **Abstract (best sentence)** | _\"{abs_sent}\"_ |\n"
        f"| **Body chunk (best sentence)** | _\"{body_sent}\"_ |\n"
        f"| **Rank of gold** | abstract-only **{row['rank_abs_insuff']}** &nbsp;→&nbsp; "
        f"+chunks **{row['rank_chk_insuff']}** |\n"
    )


def render_ex3_latex(qid: int) -> str:
    row = wide_ex3.filter(pl.col("qid") == qid).row(0, named=True)
    pmid = row["insufficient_pmids"][0]
    idx = row["gold_pmids"].index(pmid)
    title = short_title(row["gold_titles"][idx], 95)
    abstract = row["gold_abstracts"][idx] or ""
    abs_sent = best_sentence(row["query_text"], abstract)
    body_text = ""
    for ch in top_chunks_for_pmid(qid, pmid, k=5):
        if ch.get("type") == "body" and ch.get("text"):
            body_text = ch["text"]
            break
    body_sent = best_sentence(row["query_text"], body_text)
    return (
        "\\begin{table}[t]\n"
        "\\centering\n"
        "\\caption{Qualitative example illustrating the gain from full-text chunks on an "
        "abstract-insufficient claim. The cited abstract does not contain the specific cue "
        "the curator claim depends on; a body-text chunk of the same article does, and "
        "adding chunks to the corpus moves the article from outside the top-K into the top-K. "
        "Ranks are post-rerank positions under BGE-reranker-v2-m3.}\n"
        "\\label{tab:fulltext_example}\n"
        "\\scriptsize\n"
        "\\setlength{\\tabcolsep}{6pt}\n"
        "\\begin{tabular}{@{}p{2.4cm}p{8.6cm}@{}}\n"
        "\\toprule\n"
        f"Curator claim & {_esc(row['query_text'])} \\\\\n"
        f"Gold PMID {pmid} & \\emph{{{_esc(title)}}} \\\\\n"
        f"Abstract (best sentence) & \\small\\itshape ``{_esc(abs_sent)}'' \\\\\n"
        f"Body chunk (best sentence) & \\small\\itshape ``{_esc(body_sent)}'' \\\\\n"
        "\\midrule\n"
        "\\multicolumn{2}{@{}l@{}}{Rank of gold under BGE-reranker-v2-m3: "
        f"abstract-only corpus = {row['rank_abs_insuff']}, "
        f"+ chunks = {row['rank_chk_insuff']}.}} \\\\\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\end{table}"
    )

### Emit previews

Markdown for the notebook (easy to scan); LaTeX for the manuscript.

In [60]:
def _hr(title: str) -> str:
    return f"\n---\n\n## {title}\n"


print(_hr("Ex.1a — Lexical signal lost (Option 2: keep qid 393, add Why annotation)"))
if SELECTED_EX1A is not None:
    print(render_ex1_markdown(SELECTED_EX1A, "Lexical signal lost"))

print(_hr("Ex.1a — Lexical signal lost (Option 1: swap to qid 535 with cleaned query + Why)"))
if ALT_EX1A is not None:
    print(render_ex1_markdown(ALT_EX1A, "Lexical signal lost"))

print(_hr("Ex.1b — Semantic rescue (qid 1616)"))
if SELECTED_EX1B is not None:
    print(render_ex1_markdown(SELECTED_EX1B, "Semantic rescue"))

print(_hr("Ex.2 — Gene-aware query expansion (qid 313)"))
if SELECTED_EX2 is not None:
    print(render_ex2_markdown(SELECTED_EX2))

print(_hr("Ex.3 — Full-text retrieval (qid 534)"))
if SELECTED_EX3 is not None:
    print(render_ex3_markdown(SELECTED_EX3))


---

## Ex.1a — Lexical signal lost (Option 2: keep qid 393, add Why annotation)

### Ex.1 — Lexical signal lost &nbsp;(qid 393)

| | |
|---|---|
| **Query** | DymA also localizes to the phagosome and the cleavage furrow. |
| **Gold** | PMID 23679940 &nbsp; *Dynamin contributes to cytokinesis by stabilizing actin filaments in the contractile ring.*<br>&nbsp;&nbsp;&nbsp;&nbsp;"DymA and DlpA were associated with actin filaments at the furrow."<br>&nbsp;&nbsp;&nbsp;&nbsp;ranks: BM25 **2**, MS-MARCO **60**, Gemma **3** (BGE-m3 3, MedCPT 5) |
| **Competitor** | PMID 7820857 &nbsp; *Differential localization of alpha-actinin and the 30 kD actin-bundling protein in the...*<br>&nbsp;&nbsp;&nbsp;&nbsp;"The 30 kD protein is concentrated in the cleavage furrow of dividing cells, while enhanced staining for alpha-actinin is not apparent in this region."<br>&nbsp;&nbsp;&nbsp;&nbsp;ranks: BM25 **5**, MS-MARCO **1**, Gemma **6** |
| **Why** | _MS-MARCO promotes a topically-related Dicty paper that s

In [61]:
# LaTeX output for paper.tex. Ex.1 packs both example blocks into one table.
ex1_pairs = []
if SELECTED_EX1A is not None:
    ex1_pairs.append((SELECTED_EX1A, "Lexical signal lost"))
if SELECTED_EX1B is not None:
    ex1_pairs.append((SELECTED_EX1B, "Semantic rescue"))
if ex1_pairs:
    print("% --- Ex.1: Reranker behavior ---")
    print(render_ex1_latex(ex1_pairs))
    print()
if SELECTED_EX2 is not None:
    print("% --- Ex.2: Query expansion ---")
    print(render_ex2_latex(SELECTED_EX2))
    print()
if SELECTED_EX3 is not None:
    print("% --- Ex.3: Full-text retrieval ---")
    print(render_ex3_latex(SELECTED_EX3))

% --- Ex.1: Reranker behavior ---
\begin{table}[t]
\centering
\caption{Qualitative examples illustrating reranker behavior on two curator-claim queries. Each block shows the gold article and one competitor article (the top-ranked non-gold article under the weaker reranker for 'Lexical signal lost', or under BM25 for 'Semantic rescue'). Ranks are positions of each article in the final ranking; lower is better. Quoted lines are best-matching sentences from each article's abstract.}
\label{tab:rerank_examples}
\scriptsize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{@{}p{8.0cm}rrr@{}}
\toprule
Article & BM25 & MS-MARCO & Gemma \\
\midrule
\multicolumn{4}{@{}l@{}}{\emph{Lexical signal lost.} \textit{Query:} DymA also localizes to the phagosome and the cleavage furrow.} \\
Gold PMID 23679940 --- Dynamin contributes to cytokinesis by stabilizing actin filaments in the... & 2 & 60 & 3 \\
\multicolumn{4}{@{}l@{}}{\quad\small\itshape ``DymA and DlpA were associated with actin filaments at the fu